# Cellprofiler环境搭建

In [1]:
import os
import sys

# 设置正确的 Java 路径（基于你的终端输出）
os.environ['JAVA_HOME'] = '/home/fcl/.conda/envs/cp_env'
os.environ['PATH'] = '/home/fcl/.conda/envs/cp_env/bin:' + os.environ.get('PATH', '')

# 验证 Java 现在是否可用
import subprocess
result = subprocess.run(['java', '-version'], capture_output=True, text=True)
print("Java 版本:")
print(result.stderr)

# 现在导入 CellProfiler
import cellprofiler_core.preferences
import cellprofiler_core.utilities.java as cp_java
import cellprofiler_core.pipeline

print("\n✅ 所有模块导入成功")

# 启动 JVM
cp_java.start_java()
print("✅ JVM 启动成功")

Java 版本:
openjdk version "11.0.29-internal" 2025-10-21
OpenJDK Runtime Environment (build 11.0.29-internal+0-adhoc..src)
OpenJDK 64-Bit Server VM (build 11.0.29-internal+0-adhoc..src, mixed mode)


✅ 所有模块导入成功
✅ JVM 启动成功


In [2]:
import cellprofiler_core.preferences
import cellprofiler_core.utilities.java
import cellprofiler_core.pipeline
import cellprofiler_core.workspace
import cellprofiler_core.measurement
import pathlib

cellprofiler_core.preferences.set_headless()
cellprofiler_core.preferences.set_plugin_directory("")
cellprofiler_core.utilities.java.start_java()

In [3]:
import logging
import datetime
import glob
# 训练logger
def run_log():
    logger = logging.getLogger()
    
    for hdlr in range(len(logger.handlers)):
        logger.removeHandler(logger.handlers[-1])

    # 设置日志格式，创建logger对象，设置StreamHandler日志等级
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    logger.setLevel(logging.INFO)
    
    sh = logging.StreamHandler()
    sh.setLevel(logging.INFO)
    sh.setFormatter(formatter)
    logger.addHandler(sh)

    # 创建Filehandler对象，写入.log文件，设置写入文件的日志等级
    run_log = logging.FileHandler('./%s.log' % datetime.datetime.now(),'a',encoding='utf-8')
    run_log.setLevel(logging.INFO)
    run_log.setFormatter(formatter)

    # 加载文件到logger对象中
    logger.addHandler(run_log)

    # 测试
    logging.info('info')
    logging.warning('warning')


# 载入pipeline文件，修改pipeline并添加需要处理的图像

In [4]:
import cellprofiler
import cellprofiler.modules

In [5]:
pipeline = cellprofiler_core.pipeline.Pipeline()
pipeline.load("pipeline_3D-1115.cppipe")

In [6]:
file_list = list(pathlib.Path('/').absolute().glob('data1/fcl/sciAbuchong/FCL-20260508-RAW-3D-SIMFN2-2/*.tiff'))
files = [file.as_uri() for file in file_list]
pipeline.read_file_list(files)

In [7]:
#pipeline.modules()

In [8]:
#[i.to_dict() for i in pipeline.modules()[24].settings()]

In [9]:
#pipeline.modules()[2].setting(11).set_value('0.6')
#pipeline.modules()[2].setting(12).set_value('0.6')
#pipeline.modules()[6].setting(3).set_value('No')
#pipeline.modules()[6].setting(9).set_value('8')
pipeline.modules()[7].setting(10).set_value('Elsewhere...|/data1/fcl/sciAbuchong/FCL-20260508-RAW-3D-SIMFN2-2/overlayimages')
pipeline.modules()[9].setting(10).set_value('Elsewhere...|/data1/fcl/sciAbuchong/FCL-20260508-RAW-3D-SIMFN2-2/overlayimages')
pipeline.modules()[19].setting(10).set_value('Elsewhere...|/data1/fcl/sciAbuchong/FCL-20260508-RAW-3D-SIMFN2-2/overlayimages')
pipeline.modules()[23].setting(8).set_value('Elsewhere...|/data1/fcl/sciAbuchong/FCL-20260508-RAW-3D-SIMFN2-2/CSV')

# 构建workspace

In [10]:
m = cellprofiler_core.measurement.Measurements()

workspace = cellprofiler_core.workspace.Workspace(pipeline,None,None,None,m,None)
pipeline.prepare_run(workspace)

True

# 运行pipeline

In [11]:
from cellprofiler_core.analysis.event import Started
from cellprofiler_core.analysis._runner import Runner


def fake_event_listener(evt):
    if isinstance(evt, Started):
        return

In [ ]:
import uuid

  
run_log()

num_workers = 32

runner = Runner(analysis_id=uuid.uuid1().hex,
                pipeline=pipeline,
                initial_measurements_buf=workspace.measurements.file_contents(),
                event_listener=fake_event_listener)

runner.start(num_workers)


2026-05-14 16:48:48,846 - INFO - info
2026-05-14 16:48:48,847 - WARNING - warning


# 停止运行

In [14]:
runner.stop_workers()